# Ensembling SGTs — Random SG Forests

`RandomSGForestClassifier` / `RandomSGForestRegressor` bag many SGTs, each trained on a bootstrap resample with a random feature subset per split, and average their predictions. This usually beats a single tree at the cost of interpretability. This notebook shows the accuracy gain and the knobs that control it.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sgtlearn import SGTClassifier, RandomSGForestClassifier

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

tree = SGTClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr)
forest = RandomSGForestClassifier(
    n_estimators=50, max_depth=3, max_features="sqrt", random_state=0, n_jobs=-1
).fit(X_tr, y_tr)

print(f"single tree  test={tree.score(X_te, y_te):.3f}")
print(f"forest (50)  test={forest.score(X_te, y_te):.3f}")

single tree  test=0.877
forest (50)  test=0.959


## What to notice

The forest averages 50 decorrelated trees, so its test accuracy is typically higher and more stable than a single tree of the same depth. Compare the two printed numbers.

## The knobs

- `n_estimators` — number of trees. More = better, with diminishing returns.
- `bootstrap` — resample rows per tree (True) for diversity; `max_samples` sets the resample size.
- `max_features` — features considered per split (`"sqrt"`, `"log2"`, int, float). Lower = more diverse trees.
- `n_jobs` — parallel tree fitting (`-1` = all cores).

In [2]:
# More trees, then a smaller per-split feature set for extra diversity.
for n in (5, 25, 100):
    f = RandomSGForestClassifier(
        n_estimators=n, max_depth=3, max_features="sqrt", random_state=0, n_jobs=-1
    ).fit(X_tr, y_tr)
    print(f"n_estimators={n:3d}: test={f.score(X_te, y_te):.3f}")

# bootstrap + max_samples control resampling.
f = RandomSGForestClassifier(
    n_estimators=50, max_depth=3, bootstrap=True, max_samples=0.6,
    max_features="log2", random_state=0, n_jobs=-1
).fit(X_tr, y_tr)
print(f"bootstrap 60% rows, log2 feats: test={f.score(X_te, y_te):.3f}")

n_estimators=  5: test=0.912
n_estimators= 25: test=0.953
n_estimators=100: test=0.965
bootstrap 60% rows, log2 feats: test=0.953


## How does it compare to a classic random forest?

scikit-learn's `RandomForestClassifier` bags axis-aligned CART trees (each node is a single `x < t` threshold); `RandomSGForestClassifier` bags SGTs whose nodes apply a shape function to a feature. Compared at matched settings on the same split:

In [3]:
from sklearn.ensemble import RandomForestClassifier

sg = RandomSGForestClassifier(
    n_estimators=50, max_depth=3, max_features="sqrt", random_state=0, n_jobs=-1
).fit(X_tr, y_tr)
rf = RandomForestClassifier(
    n_estimators=50, max_depth=3, max_features="sqrt", random_state=0, n_jobs=-1
).fit(X_tr, y_tr)

print(f"RandomSGForest        test={sg.score(X_te, y_te):.3f}")
print(f"sklearn RandomForest  test={rf.score(X_te, y_te):.3f}")

RandomSGForest        test=0.959
sklearn RandomForest  test=0.947


At matched depth and tree count the two forests are in the same ballpark. Here the SGT forest edges the classic forest, because each SGT node's shape function captures non-linear structure within a feature that an axis-aligned CART split needs extra depth to reach — so SGT forests can match or beat a classic forest at **shallower** depth. The margin is dataset- and depth-dependent; at larger `max_depth` a classic forest often catches up.

For a single interpretable model instead, see the [quickstart](../quickstart.rst) and [structure vs. accuracy](structure-and-accuracy.ipynb). TAO also refines every tree in a forest — see [TAO › Forests](tao.ipynb).